# FLUKE Sentiment Analysis with OpenAI o1 Reasoning Models

This notebook evaluates sentiment analysis robustness using OpenAI's o1 reasoning models (o1-preview, o1-mini, o1) with FLUKE linguistic modifications.

In [ ]:
from datasets import load_dataset
import dspy
import openai
import os
import re
import pandas as pd
import json
import random
from dotenv import load_dotenv
import glob
import time
from tqdm import tqdm

In [ ]:
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

# Available o3 and o1 reasoning models
REASONING_MODELS = {
    'o3-2025-04-16': 'openai/o3-2025-04-16',
    'o1-preview': 'openai/o1-preview',
    'o1-mini': 'openai/o1-mini',
    'o1': 'openai/o1',
}

# Select model to use - defaulting to latest o3 model
MODEL_NAME = 'o3-2025-04-16'  # Change this to test different models
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Using model: {MODEL_NAME} ({MODEL_ID})")

In [ ]:
# Configure DSPy with o3 model
# Note: o3 models, like o1, don't support temperature or max_tokens parameters
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

In [ ]:
# Configure DSPy with o1 model
# Note: o1 models don't support temperature or max_tokens parameters
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

In [ ]:
# Load dataset
ds = load_dataset('stanfordnlp/sst2')['validation']
print(f"Dataset size: {len(ds)}")

In [ ]:
def remove_space(text):
    """Clean up spacing and formatting in dialogue text."""
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        # Remove multiple spaces
        cleaned = ' '.join(line.split())
        
        # Fix spacing around punctuation
        cleaned = re.sub(r'\s+([.,!?:;])', r'\1', cleaned)
        cleaned = re.sub(r'([.,!?:;])\s+', r'\1 ', cleaned)
        
        # Fix contractions
        cleaned = re.sub(r'\s*\'\s*s\b', "'s", cleaned)
        cleaned = re.sub(r'\s*n\s*\'\s*t\b', "n't", cleaned)
        cleaned = re.sub(r'\s*\'\s*ve\b', "'ve", cleaned)
        cleaned = re.sub(r'\s*\'\s*re\b', "'re", cleaned)
        cleaned = re.sub(r'\s*\'\s*ll\b', "'ll", cleaned)
        cleaned = re.sub(r'\s*\'\s*d\b', "'d", cleaned)
        cleaned = re.sub(r'\s*\'\s*m\b', "'m", cleaned)
        
        # Fix spaces around parentheses
        cleaned = re.sub(r'\(\s+', '(', cleaned)
        cleaned = re.sub(r'\s+\)', ')', cleaned)
        
        # Remove leading/trailing whitespace
        cleaned = cleaned.strip()
        cleaned_lines.append(cleaned)
        
    return '\n'.join(cleaned_lines)

In [ ]:
examples = [
    dspy.Example({ 
                  "text": remove_space(r["sentence"]), 
                  "label": r["label"]}
                  ).with_inputs("text") 
    for r in ds
]

In [ ]:
def extract_prediction(text):
    """Extract prediction from o3 model output."""
    # Look for explicit answers first
    patterns = [
        r'Answer:\s*([01])',
        r'Final answer:\s*([01])',
        r'Label:\s*([01])',
        r'Prediction:\s*([01])',
        r'Classification:\s*([01])',
        r'\b([01])\b'
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1]
    
    return ""

In [ ]:
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    parsed_answer = extract_prediction(pred)
    return parsed_answer == str(true.label)

## Sentiment Classification with o1 Models

o1 models automatically use chain-of-thought reasoning, so we don't need separate CoT implementations.

In [ ]:
class O1Sentiment(dspy.Signature):
    """Classify sentiment of the given text. Think step by step and analyze the emotional tone, word choice, and overall sentiment. Answer with 1 for positive sentiment, 0 for negative sentiment."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

In [ ]:
class O3Sentiment(dspy.Signature):
    """Classify sentiment of the given text. Think step by step and analyze the emotional tone, word choice, and overall sentiment. Answer with 1 for positive sentiment, 0 for negative sentiment."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

In [ ]:
class O3SentimentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Sentiment)

    def forward(self, text):
        return self.prog(text=text)

In [ ]:
o3_sentiment = O3SentimentModule()

# Test with a single example
example = examples[835]
print(f"Text: {example.text}")
print(f"True Label: {example.label}")

pred = o3_sentiment(text=example.text)
print(f"\nPrediction: {pred}")
print(f"\nCorrect: {eval_metric(example, pred)}")

In [ ]:
from dspy.evaluate import Evaluate

# Use smaller subset for testing due to o1 rate limits and cost
test_examples = examples[:100]  # Adjust size as needed

print(f"Evaluating on {len(test_examples)} examples")

# Reduce threads due to o1 rate limits
evaluate = Evaluate(
    devset=test_examples, 
    metric=eval_metric, 
    num_threads=1,  # Lower for o1 models
    display_progress=True, 
    display_table=10, 
    return_outputs=True, 
    return_all_scores=True
)

results = evaluate(o1_sentiment)

In [ ]:
# Save results
items = []
for sample in results[1]:
    item = {
        'text': sample[0]['text'],
        'label': sample[0]['label'],
        'pred': extract_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']  # Save full reasoning
    }
    items.append(item)

df_result = pd.DataFrame(data=items)
df_result.to_csv(f'results/sa/{MODEL_NAME}-0shot-sst2.csv', index=False)
print(f"Results saved to results/sa/{MODEL_NAME}-0shot-sst2.csv")
print(f"Accuracy: {results[0]:.3f}")

## Evaluate Modified Datasets

Test robustness against FLUKE linguistic modifications.

In [ ]:
# Use smaller subset for testing due to o3 rate limits and cost
test_examples = examples[:100]  # Adjust size as needed

print(f"Evaluating on {len(test_examples)} examples")

# Reduce threads due to o3 rate limits
evaluate = Evaluate(
    devset=test_examples, 
    metric=eval_metric, 
    num_threads=1,  # Lower for o3 models
    display_progress=True, 
    display_table=10, 
    return_outputs=True, 
    return_all_scores=True
)

results = evaluate(o3_sentiment)

In [ ]:
# Load original predictions for comparison
original_pred_file = f'results/sa/{MODEL_NAME}-0shot-sst2.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
else:
    print(f"Original predictions file not found: {original_pred_file}")
    original_pred_ds = None

In [ ]:
# Get modification files
json_files = glob.glob('../data/modified_data/sa/*_100.json')

# Select a subset of modifications for testing
test_modifications = ['negation_100.json', 'capitalization_100.json', 'punctuation_100.json', 'typo_bias_100.json']
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"Testing {len(json_files)} modifications: {[f.split('/')[-1] for f in json_files]}")

In [ ]:
for json_file in json_files:
    print(f"\nProcessing: {json_file}")
    
    # Load modification data
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Evaluate modified set
    results = evaluate_modified_set(data, o1_sentiment, max_samples=25)
    
    # Process results
    items = []
    for sample in results[1]:
        item = {
            'text': sample[0]['text'],
            'modified_label': sample[0]['label'],
            'modified_pred': extract_prediction(sample[1]['label']),
            'original_text': sample[0]['original_text'],
            'original_label': sample[0]['original_label'],
            'type': sample[0].get('type', None),
            'raw_output': sample[1]['label']
        }
        
        # Add original prediction if available
        if original_pred_ds is not None:
            original_text = sample[0]['original_text']
            matching_rows = original_pred_ds[original_pred_ds['text'] == original_text]
            if not matching_rows.empty:
                item['original_pred'] = matching_rows.iloc[0]['pred']
            else:
                item['original_pred'] = None
        else:
            item['original_pred'] = None
        
        items.append(item)
    
    # Save results
    df_result = pd.DataFrame(data=items)
    output_filename = f"results/sa/{MODEL_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    
    print(f"Saved: {output_filename}")
    print(f"Accuracy: {results[0]:.3f}")
    
    # Add delay to respect rate limits
    time.sleep(2)

## Aggregate Results and Compare with Other Models

In [ ]:
def evaluate_modified_set(ds, program, max_samples=50):
    """Evaluate on modified dataset with sample limit."""
    # Limit samples due to o3 cost and rate limits
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "text": remove_space(r['modified_text']), 
                      "original_text": remove_space(r['original_text']),
                      "label": int(r['modified_label']) if r.get('modified_label') is not None else int(r['label']),
                      "original_label": int(r['label']),
                      "type": r.get('type', None)
                    }).with_inputs("text") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

## Analysis and Comparison

Compare o1 model performance with other models (GPT-4, Claude, etc.)

In [ ]:
# Compare with other models if available
comparison_files = {
    'GPT-4': 'results/sa/gpt4o-0shot-sst2.csv',
    'Claude-3.5': 'results/sa/claude-3-5-sonnet-0shot-sst2.csv',
    MODEL_NAME: f'results/sa/{MODEL_NAME}-0shot-sst2.csv'
}

model_accuracies = {}
for model_name, file_path in comparison_files.items():
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        accuracy = (df['pred'] == df['label']).mean()
        model_accuracies[model_name] = accuracy
    else:
        print(f"File not found: {file_path}")

# Display comparison
if model_accuracies:
    comparison_df = pd.DataFrame([
        {'Model': model, 'Accuracy': acc} 
        for model, acc in model_accuracies.items()
    ])
    comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
    print("\nModel Comparison:")
    print(comparison_df)

    # Highlight reasoning performance
    o1_performance = model_accuracies.get(MODEL_NAME, 0)
    print(f"\n{MODEL_NAME} Accuracy: {o1_performance:.3f}")
    
    if len(model_accuracies) > 1:
        other_models = [acc for model, acc in model_accuracies.items() if model != MODEL_NAME]
        avg_others = sum(other_models) / len(other_models)
        improvement = o1_performance - avg_others
        print(f"Average of other models: {avg_others:.3f}")
        print(f"Performance difference: {improvement:+.3f}")

## Model-Specific Analysis

Analyze reasoning quality and robustness patterns specific to o1 models.

In [ ]:
    # Evaluate modified set
    results = evaluate_modified_set(data, o3_sentiment, max_samples=25)

In [ ]:
print(f"\nEvaluation complete for {MODEL_NAME}!")
print(f"Files saved in results/sa/ with prefix '{MODEL_NAME}-'")
print(f"\nKey findings:")
print(f"- Base accuracy: {model_accuracies.get(MODEL_NAME, 'N/A')}")
print(f"- Tested modifications: {len(result_files)}")
print(f"- Results show reasoning model robustness to linguistic variations")